# 2.3 Segmentation — Full-Stack Test (510 planes)

Full-stack QC of the 2D Sauvola segmentation produced in `2_1_segmentation.ipynb`
(`savola_3D_image`, per-plane, window 21, k=0.15). This notebook **reads the saved segmentation**
(no recomputation) and QCs it per depth zone from `zone_samples.csv`, reporting **mean ± SD over
each zone's `z_start:z_end` range** — the same style as `1_2_preprocessing_full_stack_test.ipynb`.

Two metrics per zone:
- **2D component count** — connected components per plane (`count_connexin_plaques`). These are
  cross-sections, not unique 3D plaques.
- **Foreground fraction** — fraction of pixels segmented per plane.


In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from skimage import io

BASE_DIR = Path.cwd().parent
VMAX_PERCENTILE = 99.5  # display-only contrast cap for grayscale previews (matches 1_2 / 2_1)

def disp_vmax(img):
    return np.percentile(img, VMAX_PERCENTILE)

sys.path.append(str(BASE_DIR / 'src'))
from quantification import count_connexin_plaques

## 0. Load zone samples

In [ ]:
zone_samples_path = BASE_DIR / 'data' / 'zone_samples.csv'
zone_samples = pd.read_csv(zone_samples_path)
print(f"Loaded {len(zone_samples)} zones from {zone_samples_path}")
zone_samples

## 1. Load deconvolved image + segmentation

`seg_path` should point at the segmentation you want to QC — adjust it if you saved different
Sauvola parameters in `2_1_segmentation.ipynb`.

In [ ]:
deconv_path = BASE_DIR / 'data' / 'preprocessed' / 'background_removed_and_deconvolved_img.tiff'
seg_path = BASE_DIR / 'data' / 'segmented' / 'binary_sauvola2d_w21_k0_15.tiff'

deconvolved = io.imread(deconv_path)
seg = io.imread(seg_path) > 0  # ubyte 0/255 -> bool foreground

print(f"Deconvolved: shape={deconvolved.shape}, dtype={deconvolved.dtype}")
print(f"Segmentation: shape={seg.shape}, dtype=bool (from {seg_path.name})")

if seg.shape != deconvolved.shape:
    print(f"WARNING: segmentation shape {seg.shape} != deconvolved shape {deconvolved.shape}")

n_planes = seg.shape[0]
oor = zone_samples[~zone_samples['sample_plane'].between(0, n_planes - 1)]
print("All sample planes in range." if not len(oor) else f"WARNING: out-of-range planes:\n{oor}")

## 2. QC helpers

In [ ]:
def plot_zone_overlay_grid(image, binary, zone_df, ncols=4):
    """Per zone: the sample_plane grayscale (percentile vmax) with the segmentation
    boundary drawn as a red contour."""
    n = len(zone_df)
    ncols = min(ncols, n)
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 4 * nrows), squeeze=False)
    for i, (_, row) in enumerate(zone_df.iterrows()):
        ax = axes[i // ncols, i % ncols]
        p = int(row['sample_plane'])
        ax.imshow(image[p], cmap='gray', vmax=disp_vmax(image[p]))
        ax.contour(binary[p], [0.5], colors='r', linewidths=0.4)
        ax.set_title(f"{row['zone_label']} (plane {p})")
        ax.axis('off')
    for j in range(n, nrows * ncols):
        axes[j // ncols, j % ncols].axis('off')
    fig.suptitle('Segmentation overlay — sample plane per zone')
    plt.tight_layout()
    plt.show()


def format_mean_sd(mean, sd):
    if np.isnan(mean):
        return "n/a"
    if np.isnan(sd):
        return f"{mean:.3g}"
    return f"{mean:.3g} ± {sd:.3g}"


def zone_segmentation_stats(binary, zone_df):
    """Per zone, over every plane in [z_start, z_end): mean +/- SD of the 2D component count
    and of the foreground area fraction."""
    n_planes = binary.shape[0]
    rows = []
    for _, row in zone_df.iterrows():
        z0, z1 = int(row['z_start']), min(int(row['z_end']), n_planes)
        counts, fracs = [], []
        for z in range(z0, z1):
            counts.append(int(count_connexin_plaques(binary[z])[0]))
            fracs.append(float(binary[z].mean()))
        counts, fracs = np.asarray(counts), np.asarray(fracs)
        rows.append({
            'zone_number': row['zone_number'],
            'zone_label': row['zone_label'],
            'z_range': f"{int(row['z_start'])}:{int(row['z_end'])}",
            'count_mean': counts.mean() if counts.size else np.nan,
            'count_sd': counts.std(ddof=1) if counts.size > 1 else np.nan,
            'fgfrac_mean': fracs.mean() if fracs.size else np.nan,
            'fgfrac_sd': fracs.std(ddof=1) if fracs.size > 1 else np.nan,
        })
    return pd.DataFrame(rows)

## 3. Per-zone overlay QC

In [ ]:
plot_zone_overlay_grid(deconvolved, seg, zone_samples)

## 4. Per-zone segmentation stats (mean ± SD over each zone's z-range)

In [ ]:
seg_stats = zone_segmentation_stats(seg, zone_samples)

seg_display = pd.DataFrame({
    'zone_label': seg_stats['zone_label'],
    'z_range': seg_stats['z_range'],
    'components/plane (mean ± SD)': [format_mean_sd(m, s) for m, s in zip(seg_stats['count_mean'], seg_stats['count_sd'])],
    'foreground fraction (mean ± SD)': [format_mean_sd(m, s) for m, s in zip(seg_stats['fgfrac_mean'], seg_stats['fgfrac_sd'])],
})
print("2D connected components and foreground fraction per plane, aggregated over each zone's z-range:")
seg_display

## 5. Full-stack per-plane trend

Exposes depth-dependent dropout across the whole stack.

In [ ]:
counts = np.array([count_connexin_plaques(seg[z])[0] for z in range(n_planes)])
fgfrac = np.array([seg[z].mean() for z in range(n_planes)])

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
ax1.bar(range(n_planes), counts, width=1.0, color='steelblue', edgecolor='none')
ax1.set_ylabel('2D components / plane')
ax1.set_title('Segmentation across the full stack')
ax1.grid(axis='y', alpha=0.3)
ax2.plot(range(n_planes), fgfrac, color='indianred')
ax2.set_xlabel('Plane index (z)')
ax2.set_ylabel('Foreground fraction')
ax2.grid(alpha=0.3)

# shade zone boundaries for context
for _, row in zone_samples.iterrows():
    ax1.axvline(int(row['z_start']), color='0.7', lw=0.5)
    ax2.axvline(int(row['z_start']), color='0.7', lw=0.5)
plt.tight_layout()
plt.show()

## Summary

Reads the saved full-stack 2D Sauvola segmentation from `2_1_segmentation.ipynb` and QCs it per
depth zone; it does not recompute segmentation.

- **Section 4** reports 2D component count and foreground fraction per plane as mean ± SD over
  each zone's `z_start:z_end` range.
- **Section 5** shows both metrics across the whole stack, with zone boundaries marked, to expose
  depth-dependent dropout (e.g. the near-empty deep planes seen in 2_1).

**Caveats:** component counts are 2D cross-sections per plane, **not** unique 3D plaques — use the
3D labeling in the localization notebook for true plaque counts. Segmentation parameters are
inherited from `2_1` (whatever produced `seg_path`); the overlay images show one representative
plane per zone while the tables span the full zone range.